# PEFT Fine-Tuning: Car Rental Review Summarization & Classification
Fine-tunes `google/flan-t5-base` on `amazon_us_reviews/Automotive_v1_00` using LoRA adapters.

The model is trained on two tasks simultaneously:
- **Summarization** — condense a review into its headline
- **Classification** — assign one or more categories (Vehicle Condition, Customer Service, etc.)

> **Tip:** Go to `Runtime → Change runtime type` and select **GPU** (T4) for faster training.

In [ ]:
!pip install -q "torchao>=0.16.0" transformers datasets peft accelerate huggingface_hub

In [ ]:
import time
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

## 0. Config
Mirrors `config.py`. Change these to control training behaviour.

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME    = "google/flan-t5-base"
ADAPTER_PATH  = "YOUR_HF_USERNAME/car-rental-peft-adapter"  # ← set your HF repo here

# ── Dataset ────────────────────────────────────────────────────────────────
DATASET_NAME   = "amazon_us_reviews"
DATASET_SUBSET = "Automotive_v1_00"
SUBSAMPLE_RATIO = 100  # keep 1 in every N reviews (100 = ~1% for fast iteration)

# ── Training ───────────────────────────────────────────────────────────────
MAX_STEPS    = 1       # set to None for a full training run
NUM_EPOCHS   = 1
LEARNING_RATE = 1e-3

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Categories ─────────────────────────────────────────────────────────────
CATEGORIES = [
    "Vehicle Condition",
    "Customer Service",
    "Pricing & Billing",
    "Insurance & Documents",
    "Pickup & Return",
    "App & Booking",
]

## 1. Detect Device

In [ ]:
def get_device():
    print('🚀 Initializing the LLM Pipeline...')
    print('--------------------------------------------------')
    print('✅ All required libraries imported successfully.')
    print(f'✅ PyTorch version: {torch.__version__}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✅ Processing device set to: {device.upper()}')
    print('--------------------------------------------------')
    return device

device = get_device()

## 2. Load & Build Multi-Task Dataset
Loads `amazon_us_reviews/Automotive_v1_00` and builds two training examples per review:
- **Summarization**: review body → review headline
- **Classification**: review body → matching categories

In [ ]:
# ── Keyword labeler (mirrors data_processing/labeler.py) ───────────────────
CATEGORY_KEYWORDS = {
    "Vehicle Condition": [
        "dirty", "unclean", "filthy", "damage", "damaged", "scratch", "scratched",
        "dent", "dented", "smell", "smells", "smelly", "odor", "stain", "stained",
        "mechanical", "breakdown", "broke down", "engine", "tire", "flat tire",
        "broken", "malfunction", "defect", "defective", "worn", "old car",
        "maintenance", "warning light", "check engine", "interior", "exterior",
        "cigarette", "smoke", "pet hair", "debris", "rust",
    ],
    "Customer Service": [
        "staff", "agent", "employee", "representative", "rude", "polite",
        "helpful", "unhelpful", "attitude", "customer service", "support",
        "communication", "friendly", "unfriendly", "professional", "unprofessional",
        "manager", "counter", "ignored", "dismissive", "disrespectful",
        "wait time", "long wait", "no one helped",
    ],
    "Pricing & Billing": [
        "charge", "charged", "overcharge", "fee", "fees", "hidden fee",
        "price", "expensive", "cost", "billing", "invoice", "refund",
        "deposit", "credit card", "extra charge", "surcharge", "unauthorized",
        "fraud", "scam", "money", "payment", "receipt", "tax", "overpriced",
        "misleading price", "unexpected charge",
    ],
    "Insurance & Documents": [
        "insurance", "coverage", "liability", "contract", "agreement",
        "document", "fine print", "terms", "conditions", "policy",
        "waiver", "cdw", "collision", "damage waiver", "excess",
        "deductible", "claim", "accident report", "paperwork", "sign",
    ],
    "Pickup & Return": [
        "pickup", "pick up", "pick-up", "return", "drop off", "drop-off",
        "wait", "waiting", "queue", "long wait", "delay", "delayed",
        "location", "shuttle", "bus", "airport", "station", "ready",
        "not ready", "line", "check-in", "check-out", "key",
    ],
    "App & Booking": [
        "app", "application", "website", "online", "booking", "book",
        "reservation", "confirm", "confirmation", "email", "notification",
        "crash", "error", "bug", "glitch", "slow", "update",
        "interface", "digital", "portal", "system", "platform", "login",
    ],
}

def label_review(text):
    text_lower = text.lower()
    matched = [
        cat for cat, keywords in CATEGORY_KEYWORDS.items()
        if any(kw in text_lower for kw in keywords)
    ]
    return matched if matched else ["Customer Service"]


# ── Dataset loader (mirrors data_processing/loader.py) ─────────────────────
def load_review_dataset():
    print(f"\n📥 Loading {DATASET_NAME} / {DATASET_SUBSET} from Hugging Face...")
    dataset = load_dataset(DATASET_NAME, DATASET_SUBSET, trust_remote_code=True)
    print(f"✅ Loaded {dataset['train'].num_rows:,} reviews")
    sample = dataset["train"][0]
    print(f"\n🔍 Sample:\n  Review  : {sample['review_body'][:200]}...\n  Headline: {sample['review_headline']}")
    return dataset


def build_multitask_dataset(dataset):
    print(f"\n⚙️  Building multi-task examples (1 in every {SUBSAMPLE_RATIO} reviews)...")
    categories_str = ", ".join(CATEGORIES)
    inputs, outputs = [], []

    for idx, row in enumerate(dataset["train"]):
        if idx % SUBSAMPLE_RATIO != 0:
            continue
        body     = (row.get("review_body") or "").strip()
        headline = (row.get("review_headline") or "").strip()
        if not body or not headline:
            continue

        inputs.append(f"Summarize the following car rental review.\n\n{body}\n\nSummary:")
        outputs.append(headline)

        inputs.append(
            f"Classify this car rental review into one or more of these categories: "
            f"{categories_str}.\n\nReview: {body}\n\nCategories:"
        )
        outputs.append(", ".join(label_review(body)))

    split = Dataset.from_dict({"input": inputs, "output": outputs}).train_test_split(test_size=0.1, seed=42)
    print(f"✅ {split['train'].num_rows:,} train / {split['test'].num_rows:,} validation examples")
    return split


raw_dataset       = load_review_dataset()
multitask_dataset = build_multitask_dataset(raw_dataset)

## 3. Load Model & Tokenizer

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(
        f"📊 Model Parameter Report:\n"
        f"--------------------------------------------------\n"
        f"  Total Parameters:     {total:,}\n"
        f"  Trainable Parameters: {trainable:,}\n"
        f"  Percentage Trainable: {100 * trainable / total:.4f}%\n"
        f"--------------------------------------------------"
    )

def load_tokenizer(model_name=MODEL_NAME):
    print("\n🔤 Loading Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"✅ Tokenizer loaded for {model_name}.")
    return tokenizer

def load_llm_model(device, model_name=MODEL_NAME):
    print("\n🧠 Loading the LLM...")
    target_dtype = torch.bfloat16 if device == "cuda" else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=target_dtype).to(device)
    print(f"✅ {model_name} loaded using {target_dtype}.")
    return model

tokenizer      = load_tokenizer()
original_model = load_llm_model(device)
print_number_of_trainable_model_parameters(original_model)

## 4. Tokenize Dataset
Converts `input` / `output` columns into `input_ids` / `labels` for seq2seq training.

In [ ]:
def tokenize_dataset(tokenizer, dataset):
    print("\n⚙️  Tokenizing dataset for seq2seq training...")

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input"], padding="max_length", truncation=True, max_length=512
        )
        labels = tokenizer(
            batch["output"], padding="max_length", truncation=True, max_length=128
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output"])
    print("✅ Tokenization complete.")
    return tokenized

tokenized_dataset = tokenize_dataset(tokenizer, multitask_dataset)

## 5. Inject LoRA Adapters

In [ ]:
def setup_peft_lora_model(original_model):
    print("\n🪄 Injecting LoRA Adapters (PEFT)...")
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["q", "v"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    peft_model = get_peft_model(original_model, lora_config)
    print("✅ LoRA adapters injected.")
    print_number_of_trainable_model_parameters(peft_model)
    return peft_model

peft_model = setup_peft_lora_model(original_model)

## 6. Train & Save

> Set `MAX_STEPS = None` in the Config cell for a full training run (removes the 1-step dry-run limit).

In [ ]:
def train_and_save_peft_model(peft_model, tokenizer, tokenized_datasets):
    run_dir = f"./peft-training-run-{int(time.time())}"

    training_args = TrainingArguments(
        output_dir=run_dir,
        auto_find_batch_size=True,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=1,
        max_steps=MAX_STEPS if MAX_STEPS is not None else -1,
    )

    mode = f"max_steps={MAX_STEPS}" if MAX_STEPS is not None else f"{NUM_EPOCHS} full epoch(s)"
    print(f"\n🏋️  Starting PEFT training ({mode})...")

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
    )

    trainer.train()

    trainer.model.save_pretrained("./checkpoint-local")
    tokenizer.save_pretrained("./checkpoint-local")
    print("✅ Adapter weights saved to: ./checkpoint-local")

train_and_save_peft_model(peft_model, tokenizer, tokenized_dataset)
print("\n✅ Training complete!")

## 7. Push Adapter to HuggingFace Hub

Instead of downloading the adapter files manually, push them directly to your HF account.

**One-time setup:**
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → create a token with **write** permission
2. In Colab: click the 🔑 **Secrets** icon (left sidebar) → add secret named `HF_TOKEN` → paste your token

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# ── CONFIG ─────────────────────────────────────────────────────────────────
HF_REPO = "YOUR_HF_USERNAME/car-rental-peft-adapter"  # ← change this once
# ───────────────────────────────────────────────────────────────────────────

login(token=userdata.get("HF_TOKEN"))

peft_model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"✅ Adapter pushed to: https://huggingface.co/{HF_REPO}")
print(f"   Load it anywhere with: ADAPTER_PATH = '{HF_REPO}'")